In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Set visual style for all plots
sns.set_theme(style="whitegrid")

# Load data
laws_df = pd.read_csv('Supabase Snippet Firearm Laws Time Series.csv')
incidents_df = pd.read_csv('Supabase Snippet Incident table fetch.csv')

# Quick check to ensure data loaded
print(f"Laws Data: {laws_df.shape}")
print(f"Incident Data: {incidents_df.shape}")

In [ ]:
# Calculate 2023 Law Scores by state
law_2023 = laws_df[laws_df['year'] == 2023].groupby('state')['year_frac'].sum().reset_index()
law_2023.rename(columns={'year_frac': 'law_score'}, inplace=True)

# Define the Top 5 City-Hotspots and their states
hotspot_states = {
    'District of Columbia': 'Washington',
    'California': 'Los Angeles',
    'Illinois': 'Chicago',
    'Pennsylvania': 'Philadelphia',
    'Michigan': 'Detroit'
}

# Filter and map
hotspot_policy_data = law_2023[law_2023['state'].isin(hotspot_states.keys())].copy()
hotspot_policy_data['City'] = hotspot_policy_data['state'].map(hotspot_states)
hotspot_policy_data = hotspot_policy_data.sort_values(by='law_score', ascending=False)

# National average for context
national_avg = law_2023['law_score'].mean()

# Plot
plt.figure(figsize=(10, 6))
sns.barplot(data=hotspot_policy_data, x='law_score', y='City', palette='viridis')
plt.axvline(national_avg, color='red', linestyle='--', label=f'National Avg ({national_avg:.1f})')
plt.title('Firearm Policy Strength in Top 5 City-Hotspots (2023)', fontsize=14, fontweight='bold')
plt.xlabel('State Law Score (Higher = Stricter Policies)')
plt.legend()
plt.show()

In [ ]:
# Group smaller categories into "Other" for a clean chart
top_situations = incidents_df['Situation'].value_counts().head(7)
other_count = incidents_df['Situation'].value_counts().iloc[7:].sum()
situation_data = pd.concat([top_situations, pd.Series({'Other': other_count})])

plt.figure(figsize=(8, 8))
plt.pie(situation_data, labels=situation_data.index, autopct='%1.1f%%', 
        colors=sns.color_palette('pastel'), startangle=140, pctdistance=0.85)

# Create the Donut effect
centre_circle = plt.Circle((0,0), 0.70, fc='white')
fig = plt.gcf()
fig.gca().add_artist(centre_circle)

plt.title('Breakdown of School Shooting Situations', fontsize=15, fontweight='bold')
plt.annotate('Most incidents are\nEscalations of Disputes,\nnot Mass Attacks.', 
             xy=(0,0), ha='center', fontsize=10, fontweight='bold')
plt.show()

In [ ]:
# Filter for rows where SRO presence is clearly documented as Yes or No
sro_filtered = incidents_df[incidents_df['SRO_School'].isin(['Yes', 'No'])].copy()
sro_stats = sro_filtered.groupby('SRO_School')['Number_Victims'].mean().reset_index()

plt.figure(figsize=(8, 6))
sro_plot = sns.barplot(data=sro_stats, x='SRO_School', y='Number_Victims', palette=['#3498db', '#e67e22'])

# Add labels on top of bars
for i, v in enumerate(sro_stats['Number_Victims']):
    plt.text(i, v + 0.02, f'{v:.2f}', ha='center', fontweight='bold')

plt.title('Average Casualties per Incident: With vs. Without SROs', fontsize=14, fontweight='bold')
plt.ylabel('Average Number of Victims (Killed/Wounded)')
plt.xlabel('Was an SRO Present?')
plt.show()

In [ ]:
# Grouping states by their 2023 Law Score
latest_scores = laws_df[laws_df['year'] == 2023].groupby('state')['year_frac'].sum()
strict_states = latest_scores.nlargest(10).index
permissive_states = latest_scores.nsmallest(10).index

# Map categories back to the incident data
incidents_df['policy_group'] = incidents_df['state'].apply(
    lambda x: 'Strict (Top 10)' if x in strict_states else ('Permissive (Bottom 10)' if x in permissive_states else 'Other')
)

# Count incidents per year by policy group
trend_data = incidents_df[incidents_df['policy_group'] != 'Other'].groupby(['Year', 'policy_group']).size().reset_index(name='incident_count')

plt.figure(figsize=(12, 6))
sns.lineplot(data=trend_data, x='Year', y='incident_count', hue='policy_group', marker='o', linewidth=2)
plt.title('Frequency Trends: Strict vs. Permissive Policy States', fontsize=14, fontweight='bold')
plt.ylabel('Annual Incident Count')
plt.grid(True, alpha=0.3)
plt.show()